# Smog City — Week 6: Feature Engineering

**Dataset:** `aqi_india_38cols_knn_final.csv`

## Aim
Prepare useful analysis features: parse dates, derive year/month/season where possible, encode categorical variables for modelling, and scale numeric features for comparable ranges.

This notebook keeps the original data unchanged and writes separate output files. Encoding and scaling are for analytical preparation; they do not prove a predictive model is accurate.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from google.colab import files
from IPython.display import display

pd.set_option("display.max_columns", 100)
print("Week 6 — Feature Engineering")

In [ ]:
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename, low_memory=False)
df.columns = df.columns.astype(str).str.strip().str.lower()

print("Loaded:", filename, "| shape:", df.shape)
display(df.head())

In [ ]:
# 1. Parse datetime and derive time features when available
date_candidates = ["datetime", "date", "timestamp", "time"]
date_col = next((c for c in date_candidates if c in df.columns), None)

if date_col:
    parsed = pd.to_datetime(df[date_col], errors="coerce")
    df["derived_year"] = parsed.dt.year
    df["derived_month"] = parsed.dt.month
    df["derived_day"] = parsed.dt.day
    df["derived_hour"] = parsed.dt.hour
    print("Time features derived from:", date_col)
    print("Unparseable date values:", int(parsed.isna().sum()))
else:
    print("No recognizable date/time column found; skipping date-derived features.")

# Derive season only if there is a parseable date column
if date_col:
    month = pd.to_datetime(df[date_col], errors="coerce").dt.month
    season_map = {
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Summer", 4: "Summer", 5: "Summer",
        6: "Monsoon", 7: "Monsoon", 8: "Monsoon", 9: "Monsoon",
        10: "Post-Monsoon", 11: "Post-Monsoon"
    }
    df["derived_season"] = month.map(season_map)

display(df.head())

In [ ]:
# 2. Choose a target/measurement set and inspect numeric distributions
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print("Numeric columns found:", len(numeric_cols))
print(numeric_cols)

# Save numeric feature summary
if numeric_cols:
    display(df[numeric_cols].describe().T)
    df[numeric_cols].describe().T.to_csv("Week6_Numeric_Feature_Summary.csv")

In [ ]:
# 3. One-hot encode selected categorical columns
# Limit to low-cardinality categoricals to avoid huge output files.
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
low_cardinality_cols = [c for c in categorical_cols if df[c].nunique(dropna=True) <= 30]

print("Categorical columns:", categorical_cols)
print("One-hot encoding these columns (<=30 unique values):", low_cardinality_cols)

# To keep the export manageable, use a sample if the full table is very large.
MAX_ROWS_FOR_ENCODED_EXPORT = 200000
encode_df = df.copy()
if len(encode_df) > MAX_ROWS_FOR_ENCODED_EXPORT:
    print(f"Dataset has {len(encode_df):,} rows; encoding a reproducible {MAX_ROWS_FOR_ENCODED_EXPORT:,}-row sample for demonstration.")
    encode_df = encode_df.sample(MAX_ROWS_FOR_ENCODED_EXPORT, random_state=42)

encoded_df = pd.get_dummies(
    encode_df,
    columns=low_cardinality_cols,
    dummy_na=False,
    dtype=int
)
print("Encoded shape:", encoded_df.shape)
display(encoded_df.head())
encoded_df.to_csv("Week6_Feature_Engineered_Encoded.csv", index=False)
print("Saved: Week6_Feature_Engineered_Encoded.csv")

In [ ]:
# 4. Standardize numeric columns in the working data
# Exclude derived calendar parts only if you do not want them in scaling? Here all numeric columns are scaled.
scale_cols = df.select_dtypes(include=np.number).columns.tolist()

if scale_cols:
    # Use complete numeric rows for a clear scaler demonstration.
    scale_source = df[scale_cols].replace([np.inf, -np.inf], np.nan)
    complete_rows = scale_source.dropna()
    if len(complete_rows) > 0:
        scaler = StandardScaler()
        scaled_array = scaler.fit_transform(complete_rows)
        scaled_df = pd.DataFrame(scaled_array, columns=scale_cols, index=complete_rows.index)
        print("Rows scaled:", len(scaled_df), "Numeric columns:", len(scale_cols))
        display(scaled_df.head())
        scaled_df.to_csv("Week6_Scaled_Numeric_Features.csv", index=False)
        print("Saved: Week6_Scaled_Numeric_Features.csv")
    else:
        print("No complete numeric rows available to scale.")
else:
    print("No numeric columns found; scaling skipped.")

In [ ]:
# 5. Compare original and scaled numeric ranges for the first few columns
if scale_cols and 'scaled_df' in globals() and len(scaled_df):
    compare_cols = scale_cols[:min(5, len(scale_cols))]
    comparison = pd.DataFrame({
        "original_mean": df.loc[scaled_df.index, compare_cols].mean(),
        "scaled_mean": scaled_df[compare_cols].mean(),
        "original_std": df.loc[scaled_df.index, compare_cols].std(),
        "scaled_std": scaled_df[compare_cols].std()
    })
    display(comparison.round(3))
    comparison.to_csv("Week6_Scaling_Comparison.csv")
    print("Saved: Week6_Scaling_Comparison.csv")

## Week 6 conclusion
Record which time features were created, which categorical columns were encoded, and how scaling changed numeric feature ranges. Note whether encoding was run on the full dataset or on a sample, based on the output.